# 📘 부록 01: vLLM 서버 로그와 처리량 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">읽기 자료:</strong> 노트북에서 실행할 셀이 없습니다. 강사의 RunPod 서버 로그에서 뽑은 줄을 읽습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">내 서버 로그:</strong> 6절의 명령만 RunPod 터미널 3에서 실행해 바로 봅니다.
</div>
</div>

#### **서버 통계 로그**

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM 서버가 <strong style="font-size:1.13em;color:#1f3a5f;">기본 10초마다 로그에 남기는 처리량 한 줄</strong>입니다. 그 10초 동안 서버 전체가 처리한 입력, 출력 토큰 수와, 줄을 찍는 순간의 요청 수, KV 캐시 사용률이 들어 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 벤치마크를 돌리는 동안 서버가 실제로 얼마나 일했는지 확인하고, 로그 숫자와 벤치마크 결과가 다른 이유를 설명합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 요청이 몰려 대기가 생기는지, 동시 요청을 늘린 만큼 처리량이 커지는지 서버 쪽에서 확인할 때 씁니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01 5절을 마친 뒤 읽습니다. 교안 01과 같은 노무 상담 서버의 실제 로그 줄로, 요청이 하나일 때와 여럿일 때 숫자가 왜 다른지 봅니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

**학습 목표**

1. 서버 통계 로그 한 줄의 항목을 설명합니다.
2. 입력 처리량과 출력 처리량이 무엇을 무엇으로 나눈 값인지 설명합니다.
3. 요청이 하나일 때 처리량이 작고 동시 요청이 많을 때 커지는 이유를 로그로 설명합니다.
4. 로그로 구한 출력 토큰 수를 벤치마크 결과 JSON의 출력 토큰 수와 맞춰 봅니다.
5. 내 서버의 통계 줄을 RunPod 터미널에서 바로 봅니다.

</div>

## 1. 로그 한 줄의 항목

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버는 기본 10초마다 아래 같은 한 줄을 찍습니다. 앞 줄을 찍은 때부터 이 줄을 찍을 때까지의 시간을 <strong style="font-size:1.13em;color:#1f3a5f;">구간</strong>이라고 부릅니다.</p>
</div>

```text
187: (APIServer pid=34193) INFO 10-30 08:01:46 [loggers.py:335] Engine 000: Avg prompt throughput: 904.5 tokens/s, Avg generation throughput: 408.8 tokens/s, Running: 16 reqs, Waiting: 0 reqs, GPU KV cache usage: 10.8%, Prefix cache hit rate: 93.1%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">예시 로그는 vLLM 0.32.0에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization 0.85</code>를 주어 띄운 서버에서 저장했습니다(기본값 0.92로 띄운 내 서버보다 KV 캐시가 작습니다). 수업 버전 vLLM 0.27.0도 줄을 찍는 주기와 두 처리량의 계산식이 같고(두 버전의 소스 코드로 확인), 같은 줄이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[loggers.py:310]</code>으로 찍힙니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">앞 요청과 똑같은 앞부분(이 실습은 시스템 메시지)의 계산 결과를 GPU에 남겨 두었다가 다시 쓰는 기능을 <strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(prefix caching)</strong>이라고 합니다. 남겨 둔 계산 결과는 접두어 캐시라고 부릅니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">뜻</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Avg prompt throughput</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">입력 처리량(토큰/초). 구간 동안 새로 계산한 입력 토큰 수 / 구간 길이</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Avg generation throughput</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초). 구간 동안 모든 요청이 받은 출력 토큰 수 / 구간 길이</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">줄을 찍는 순간 함께 처리 중인 요청 수. 구간 평균이 아닙니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">줄을 찍는 순간 차례를 기다리는 요청 수</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU KV cache usage</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">KV 캐시 블록 가운데 지금 요청이 쓰는 비율(순간값). 끝난 요청이 돌려준 블록은 빈 블록으로 셉니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Prefix cache hit rate</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">입력 토큰 가운데 접두어 캐시에서 찾은 비율. 구간 값이 아니라 최근 요청 1000개의 누적입니다</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">숨는 줄:</strong> 두 구간 연속으로 두 처리량이 모두 0이면 그 줄은 기본 로그에 찍히지 않습니다. 그래서 0.0 줄은 일이 끝난 바로 다음 구간에 한 번만 보입니다.</li>
</ul>
</div>

## 2. 입력 처리량과 출력 처리량

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">두 처리량은 분모가 같습니다. 둘 다 구간 길이(약 10초)로 나눕니다. 분자만 다릅니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">입력 처리량</strong> = 구간 동안 prefill로 실제로 계산한 입력 토큰 수 ÷ 구간 길이(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">캐시 제외:</strong> 접두어 캐시에서 다시 쓴 토큰은 세지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">한꺼번에 더함:</strong> 요청 하나의 입력 토큰은 그 요청의 첫 토큰이 나오는 순간 한꺼번에 더해집니다. prefill을 여러 생성 단계에 나눠 계산해도 같습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">prefill 속도가 아님:</strong> 분모가 prefill에 걸린 시간이 아니라 구간 길이입니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">출력 처리량</strong> = 구간 동안 처리 중인 모든 요청이 받은 출력 토큰 수의 합 ÷ 구간 길이(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫 토큰 포함:</strong> decode로 만든 토큰뿐 아니라 prefill 끝에 나온 첫 토큰도 셉니다. 투기적 디코딩(교안 02)이면 수용한 초안 토큰도 셉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 전체의 합:</strong> 요청 하나의 생성 속도가 아닙니다. 요청 하나의 생성 속도는 아래 식으로 구합니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">요청 하나의 생성 속도</strong> = 1000 ÷ TPOT(ms)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 하나일 때:</strong> 요청 하나만 구간 내내 돌면, 로그의 출력 처리량이 그 요청의 생성 속도와 거의 같습니다(226번 줄 63.8, 동시 요청 1의 1000 ÷ 15.34 = 65.2).</li>
</ul>
</div>

## 3. 요청 하나일 때 숫자가 작은 이유

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">첫 통계 줄은 교안 01 3절처럼 요청 하나만 보낸 구간입니다.</p>
</div>

```text
142: INFO:     127.0.0.1:52114 - "POST /v1/chat/completions HTTP/1.1" 200 OK
143: (APIServer pid=34193) INFO 10-30 08:01:26 [loggers.py:335] Engine 000: Avg prompt throughput: 432.6 tokens/s, Avg generation throughput: 22.2 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
144: (APIServer pid=34193) INFO 10-30 08:01:36 [loggers.py:335] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 요청은 입력 4,979토큰, 출력 256토큰이었고 TTFT는 526.9ms였습니다(강사 실행의 교안 01 3절 결과). 토큰 수를 처리량으로 나누면 구간 길이가 나옵니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">구간 길이</strong> = 4,979 ÷ 432.6 = 약 11.5초</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">TTFT 동안의 입력 계산 속도</strong> = 4,979 ÷ 0.527 = 약 9,450토큰/초 이상</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">11.5초:</strong> 출력으로 구해도 256 / 22.2 = 약 11.5초로 같습니다. 서버를 띄운 뒤 첫 줄이라 구간이 10초보다 깁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">432.6:</strong> 서버는 4,979토큰을 0.5초 안에 계산했지만, 로그는 이를 11.5초로 나눠 보여 줍니다. 그래서 prefill 속도가 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Running 0과 0.0 줄:</strong> 줄을 찍는 순간 요청이 이미 끝나 있었습니다. 다음 144번 줄은 일이 없어 0.0입니다.</li>
</ul>
</div>

## 4. 동시 요청이 많을 때 달라지는 값

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">생성 단계 한 번은 decode 중인 요청 모두에게 출력 토큰을 하나씩 만들어 줍니다. 그래서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>이 클수록 같은 구간에 나오는 토큰이 많습니다.</p>
</div>

```text
226: (APIServer pid=34193) INFO 10-30 08:02:26 [loggers.py:335] Engine 000: Avg prompt throughput: 111.7 tokens/s, Avg generation throughput: 63.8 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.9%, Prefix cache hit rate: 94.1%
187: (APIServer pid=34193) INFO 10-30 08:01:46 [loggers.py:335] Engine 000: Avg prompt throughput: 904.5 tokens/s, Avg generation throughput: 408.8 tokens/s, Running: 16 reqs, Waiting: 0 reqs, GPU KV cache usage: 10.8%, Prefix cache hit rate: 93.1%
214: (APIServer pid=34193) INFO 10-30 08:01:56 [loggers.py:335] Engine 000: Avg prompt throughput: 542.3 tokens/s, Avg generation throughput: 571.9 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 94.0%
412: (APIServer pid=34193) INFO 10-30 08:03:56 [loggers.py:335] Engine 000: Avg prompt throughput: 1259.5 tokens/s, Avg generation throughput: 1032.9 tokens/s, Running: 20 reqs, Waiting: 0 reqs, GPU KV cache usage: 13.2%, Prefix cache hit rate: 95.1%
415: (APIServer pid=34193) INFO 10-30 08:04:06 [loggers.py:335] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 182.4 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 95.1%
478: (APIServer pid=34193) INFO 10-30 08:04:16 [loggers.py:335] Engine 000: Avg prompt throughput: 158.4 tokens/s, Avg generation throughput: 5.7 tokens/s, Running: 8 reqs, Waiting: 43 reqs, GPU KV cache usage: 6.5%, Prefix cache hit rate: 95.1%
528: (APIServer pid=34193) INFO 10-30 08:04:26 [loggers.py:335] Engine 000: Avg prompt throughput: 2201.6 tokens/s, Avg generation throughput: 991.7 tokens/s, Running: 61 reqs, Waiting: 1 reqs, GPU KV cache usage: 37.9%, Prefix cache hit rate: 95.2%
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">강사 실행의 결과 JSON 세 개입니다. 내 서버의 값은 교안 01 5-1 테이블에서 봅니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">동시 요청 수</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TPOT(ms)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">평균 TTFT(ms)</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">63.8</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">15.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">56.5</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">16</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">611.3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">21.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">184.2</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">64</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1,064.1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">45.8</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1,212.9</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량은 늘지만 비례하지 않습니다:</strong> 16에서 64로 4배 늘리면 611.3에서 1,064.1로 약 1.7배입니다. 한 생성 단계가 느려지기 때문입니다(TPOT 21.6ms에서 45.8ms). 요청이 더 몰리면 478번 줄처럼 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Waiting</code>이 쌓입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력 처리량은 요청이 새로 들어오는 구간에만 크게 찍히고 끝에는 0이 됩니다:</strong> prefill은 요청마다 한 번이라, 동시 요청 32가 끝나 가는 415번 줄은 입력 0.0이고 출력은 182.4입니다. 시스템 메시지가 같으면 캐시에서 다시 써서 작게 찍힙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 하나의 지연은 나빠집니다:</strong> 동시 요청 1에서 64로 가면 TPOT는 15.3ms에서 45.8ms, TTFT는 56.5ms에서 1,212.9ms로 늘어납니다. 그래서 교안 01 5-3에서 지연 목표 안에서 동시 요청 수를 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">10초 평균이라 짧은 묶음은 덜 커 보입니다:</strong> 동시 요청 16의 187번 줄은 앞 4초가 비어 408.8입니다. 토큰이 빠진 것이 아니라 10초로 나눠 작게 보일 뿐입니다.</li>
</ul>
<p style="margin:6px 0;">LangChain의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">batch</code>도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency</code>만큼 요청을 함께 보내므로, 그만큼 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>이 커집니다.</p>
</div>

## 5. 로그와 벤치마크 결과 맞춰 보기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">처리량 × 구간 길이가 그 구간의 토큰 수입니다. 동시 요청 16 벤치마크는 187번 줄과 214번 줄의 두 구간(각 10초)에 걸쳐 돌았습니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">로그의 출력 토큰 수</strong> = (408.8 + 571.9) × 10 = 9,807</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">로그의 입력 토큰 수</strong> = (904.5 + 542.3) × 10 = 14,468</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">요청 하나가 캐시에서 다시 쓴 토큰</strong> = (318,596 − 14,468) ÷ 64 = 4,752</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 9,807:</strong> 결과 JSON의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">total_output_tokens</code>도 9,807입니다. 로그와 벤치마크가 같은 토큰을 셉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력 14,468:</strong> 결과 JSON의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">total_input_tokens</code> 318,596의 약 4.5%뿐입니다. 요청 64개가 4,752토큰씩 접두어 캐시에서 다시 썼기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">214번 줄의 Running 0:</strong> 5,719토큰을 만든 구간인데도 0입니다. 줄을 찍는 순간 요청이 모두 끝났을 뿐입니다.</li>
</ul>
</div>

## 6. 내 서버의 통계 줄 보기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 서버 로그는 내려받지 않고 RunPod에서 바로 봅니다. 새 터미널(RunPod 터미널 3)을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료</code> 폴더를 연 상태로 열고 아래 블록을 켜 둔 채, 터미널 2에서 벤치마크를 실행합니다(교안 01 5-1과 같은 블록).</p>
</div>

**복사해서 실행 · RunPod 터미널 3**

```bash
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다

# 서버 로그에 새로 찍히는 처리량 줄만 바로바로 보여 줍니다(멈출 때까지 계속 돕니다)
tail -f output/qwen3.5-4b_server.log | grep --line-buffered "throughput"
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">다 봤으면 터미널 3에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 멈춥니다. 요청 하나의 생성 속도는 이 줄에 없으므로 벤치마크 요약 테이블의 TPOT(교안 01 4-3)로 봅니다.</p>
</div>

## 7. 핵심 정리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력 처리량:</strong> 구간 동안 prefill로 새로 계산한 입력 토큰 수 / 구간 길이입니다. 캐시에서 다시 쓴 토큰은 빠지고, prefill 속도가 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량:</strong> 구간 동안 모든 요청이 받은 출력 토큰 수의 합 / 구간 길이입니다. 서버 전체의 값이고, 요청 하나의 생성 속도는 1000 ÷ TPOT(ms)입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">순간값과 누적값:</strong> Running, Waiting, KV 사용률은 줄을 찍는 순간의 값이고, 적중률은 최근 요청 1000개의 누적입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청이 많을 때:</strong> 출력 처리량은 커지지만 비례하지 않고, 요청 하나의 TPOT와 TTFT는 길어집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">맞춰 보기:</strong> 출력 처리량 × 구간 길이를 더하면 결과 JSON의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">total_output_tokens</code>와 같습니다.</li>
</ul>
</div>